In [1]:
!pip install ollama sentence-transformers scikit-learn
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess, time, httpx, shutil, os
import ollama

if shutil.which("ollama") is None:
    os.system("curl -fsSL https://ollama.com/install.sh | sh")

log = open("/content/ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)

for i in range(20):
    time.sleep(2)
    try:
        httpx.get("http://localhost:11434", timeout=3)
        print("server is up ✓"); break
    except Exception:
        print("waiting...", i + 1)
else:
    print("❌ السيرفر ما اشتغل — اللوق:")
    print(open("/content/ollama.log").read()[-1500:])

waiting... 1
server is up ✓


In [3]:
!ollama pull aya:8b
!ollama list


NAME            ID              SIZE      MODIFIED               
aya:8b          7ef8c4942023    4.8 GB    Less than a second ago    
qwen2.5:3b      357c53fb659c    1.9 GB    4 minutes ago             
qwen2.5:0.5b    a8b0c5157701    397 MB    9 minutes ago             


In [4]:
documents = [
    "قصة الأرنب والسلحفاة تحكي عن أرنب سريع سخر من سلحفاة بطيئة وتحدّاها في سباق، فنام الأرنب واثقًا من فوزه، بينما واصلت السلحفاة المشي بثبات حتى سبقته وفازت. والعبرة أن المثابرة تغلب الغرور.",
    "قصة الراعي الكذّاب تحكي عن راعٍ صغير كان يصرخ كذبًا بأن الذئب هاجم غنمه ليضحك على أهل القرية، وحين جاء الذئب حقًّا لم يصدّقه أحد. والعبرة أن الكذّاب لا يُصدَّق حتى لو قال الحقيقة.",
    "قصة النملة والصرصور تحكي عن نملة مجتهدة جمعت طعامها طوال الصيف استعدادًا للشتاء، بينما لها الصرصور وغنّى ولم يعمل، فلمّا جاء الشتاء جاع الصرصور وتعلّم قيمة العمل والاستعداد.",
    "قصة ليلى والذئب تحكي عن فتاة صغيرة ذهبت لزيارة جدتها المريضة في الغابة حاملة سلة طعام، فخدعها ذئب ماكر وسبقها إلى بيت الجدة، لكن صيّادًا شجاعًا أنقذها وجدتها في النهاية.",
    "قصة البطة القبيحة تحكي عن بطة صغيرة سخر منها الجميع لشكلها المختلف، فعاشت حزينة وحيدة، حتى كبرت وتبيّن أنها بجعة جميلة، فتعلّم الجميع ألا يحكموا على الآخرين من مظهرهم.",
    "قصة الأسد والفأر تحكي عن فأر صغير أطلق الأسد سراحه بعد أن أمسك به، فلمّا وقع الأسد لاحقًا في شبكة صيّاد، قرض الفأر الحبال وأنقذه. والعبرة أن المعروف لا يضيع مهما صغر صاحبه.",
    "قصة سندريلا تحكي عن فتاة طيبة عاملتها زوجة أبيها وأختاها بقسوة، فساعدتها جنّية طيبة على حضور حفل الأمير، وفقدت فردة حذائها عند منتصف الليل، فبحث عنها الأمير حتى وجدها وتزوّجها.",
]
print(f"✅ قاعدة المعرفة جاهزة — {len(documents)} قصص")

✅ قاعدة المعرفة جاهزة — 7 قصص


In [5]:
from sentence_transformers import SentenceTransformer
print("جاري تحميل النموذج...")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ النموذج جاهز")

جاري تحميل النموذج...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ النموذج جاهز


In [6]:
doc_embeddings = model.encode(documents)
print("✅ تم تحويل القصص إلى متجهات")
print("عدد القصص:", len(documents))
print("شكل المتجهات:", doc_embeddings.shape)
print("\nأول 10 أرقام من متجه القصة الأولى:")
print(doc_embeddings[0][:10])

✅ تم تحويل القصص إلى متجهات
عدد القصص: 7
شكل المتجهات: (7, 384)

أول 10 أرقام من متجه القصة الأولى:
[ 0.02340957  0.21370892 -0.04684063  0.13602725 -0.2780125  -0.06715787
  0.22856665 -0.07132655  0.10361452  0.04851928]


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(question):
    q_embedding = model.encode([question])                       # نحوّل السؤال لمتجه
    scores = cosine_similarity(q_embedding, doc_embeddings)[0]   # نقارنه بكل القصص
    best_idx = np.argmax(scores)                                 # نلقى الأعلى تشابهًا
    return best_idx, scores[best_idx]

question = "من القصة اللي فيها حيوان بطيء فاز بالسباق؟"
idx, score = search(question)
print("السؤال:", question)
print(f"\nأقرب قصة (تشابه {score:.3f}):")
print(documents[idx])

السؤال: من القصة اللي فيها حيوان بطيء فاز بالسباق؟

أقرب قصة (تشابه 0.673):
قصة الأرنب والسلحفاة تحكي عن أرنب سريع سخر من سلحفاة بطيئة وتحدّاها في سباق، فنام الأرنب واثقًا من فوزه، بينما واصلت السلحفاة المشي بثبات حتى سبقته وفازت. والعبرة أن المثابرة تغلب الغرور.


In [8]:
import ollama

MODEL = "aya:8b"

SYSTEM = """أنت مساعد يجيب عن أسئلة قصص الأطفال بالعربية الفصحى.
اعتمد فقط على النص المعطى ولا تضف أي معلومة من عندك.
أجب بجملتين كحد أقصى.
إذا لم تكن الإجابة موجودة في النص، قل فقط: لا أعرف."""

def answer(question):
    idx, score = search(question)
    context = documents[idx]

    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"القصة:\n{context}\n\nالسؤال: {question}"},
        ],
        options={"temperature": 0.2, "num_predict": 200},
    )
    return response["message"]["content"].strip()

question = "من القصة اللي فيها حيوان بطيء فاز بالسباق؟ وش العبرة منها؟"
print("السؤال:", question)
print("\nالإجابة:")
print(answer(question))

السؤال: من القصة اللي فيها حيوان بطيء فاز بالسباق؟ وش العبرة منها؟

الإجابة:
الحيوان البطئ الذي فاز بالسباق هو السلحفاة، والعبرة من القصة هي أن المثابرة والتزمّن تغلب الغرور والثقة الزائدة.


In [9]:
!nvidia-smi | head -12
print("MODEL =", MODEL)
!ollama ps

Sun Jul 26 20:51:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             34W /   70W |    5935MiB /  15360MiB |     82%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----